# Multi-Hop Retrieval — Train Model 1 (ComplementEncoder)

**Before running:**
1. Settings → Accelerator → **T4 GPU**
2. Settings → Internet → **On**
3. Your Google Drive `musique_data/` folder must contain:
   - `musique_ans_v1.0_train.jsonl`
   - `musique_ans_v1.0_dev.jsonl`
4. Folder must be shared as **Anyone with the link can view**

**What this trains:**
- `ComplementEncoder` — joint BERT `[CLS]A[SEP]B[SEP]` → B-side token matrix [n×128]
- Three-component loss: `L_content + L_orthogonal + L_chain`
  - `L_content`: complement must encode B's information
  - `L_orthogonal`: complement must NOT encode A's information (prevents shortcut)
  - `L_chain`: C-anchor contrastive — complement points toward next hop C
- Dataset: all ~26K MuSiQue hop pairs (4× more than previous 6.7K)

**Expected time: ~2 hours on T4**

In [ ]:
# ── [EDIT IF NEEDED] ───────────────────────────────────────────────────────────
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"   # musique_data folder ID
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval"
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU — must be T4 (sm_70+), not P100 (sm_60)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Settings → Accelerator → T4 GPU")

props   = torch.cuda.get_device_properties(0)
cc      = props.major
vram_gb = props.total_memory / 1e9

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"SM   : {cc}.{props.minor}")

if cc < 7:
    raise RuntimeError(
        f"GPU is P100 (sm_{cc}0) — not supported by PyTorch 2.x.\n"
        "FIX: Settings → Accelerator → change to T4 GPU"
    )
print("CUDA OK")

In [ ]:
# 2. Clone repo + install dependencies
import os

repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

# Kaggle pre-installs torch — only install missing packages
os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' gdown")
print("Dependencies ready")

In [ ]:
# 3. Download MuSiQue data from Google Drive
import os, gdown

download_dir = "/kaggle/working/drive_data"

if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(
        id=DRIVE_FOLDER_ID,
        output=download_dir,
        quiet=False,
        use_cookies=False,
    )
else:
    print("Drive data already downloaded")

print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    size = os.path.getsize(f"{download_dir}/{f}") / 1e6
    print(f"  {f:45s}  {size:7.1f} MB")

In [ ]:
# 4. Set up expected file paths
import os

drive = "/kaggle/working/drive_data"

os.makedirs(f"{WORK_DIR}/data/musique", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models",       exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache",        exist_ok=True)

# MuSiQue JSONL → symlink (avoid copying 260 MB)
for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{drive}/{fname}"
    dst = f"{WORK_DIR}/data/musique/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK ({os.path.getsize(src)/1e6:.0f} MB)")

print("\nAll paths ready")

In [ ]:
# 5. Smoke test — verify data loader and dataset size
import os
os.chdir(WORK_DIR)

# Should print:
#   [ChainDataset] ~26,675 examples (6,737 with C-anchor, 19,938 without)
os.system("python data_loader.py")

---
## Train Model 1 — ComplementEncoder

**Architecture:** joint BERT `[CLS]A[SEP]B[SEP]` → B-side token matrix [n×128] L2-norm

**Loss (three components):**
- `L_content  (w=1.0)` — complement must encode B's content
- `L_orthogonal (w=0.3)` — complement must differ from A (prevents ignoring A)
- `L_chain (w=0.5)` — C-anchor: complement points toward next hop C (only when C exists)

**Dataset:** ~26,675 MuSiQue hop pairs (all hops, not just 3/4-hop)

In [ ]:
# 6. Train Model 1 (full 3-epoch run)
import os
os.chdir(WORK_DIR)

log_file = "/kaggle/working/model1_train.log"
os.system(f"python model1_train.py 2>&1 | tee {log_file}")
print("\nTraining complete")

In [ ]:
# 7. Verify checkpoints saved correctly
import os

best  = f"{WORK_DIR}/models/model1_complement.pt"
final = f"{WORK_DIR}/models/model1_complement_final.pt"

for path in [best, final]:
    if os.path.exists(path):
        print(f"  {os.path.basename(path)}: {os.path.getsize(path)/1e6:.1f} MB  OK")
    else:
        print(f"  {os.path.basename(path)}: NOT FOUND — check training log")

# Print last 30 lines of training log
print("\n--- Last 30 lines of training log ---")
with open("/kaggle/working/model1_train.log") as f:
    lines = f.readlines()
print("".join(lines[-30:]))

---
## Save checkpoint back to Google Drive

In [ ]:
# 8. Upload model1_complement.pt back to Google Drive
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
import os

os.system("pip install -q pydrive2")

gauth = GoogleAuth()
gauth.CommandLineAuth()   # prints URL → open in browser → paste code
drive_client = GoogleDrive(gauth)

def upload_to_drive(local_path, folder_id):
    fname = os.path.basename(local_path)
    # Delete old version if it exists
    existing = drive_client.ListFile(
        {'q': f"title='{fname}' and '{folder_id}' in parents and trashed=false"}
    ).GetList()
    for old in existing:
        old.Delete()
    # Upload new
    f = drive_client.CreateFile({'title': fname, 'parents': [{'id': folder_id}]})
    f.SetContentFile(local_path)
    f.Upload()
    print(f"  Uploaded {fname} ({os.path.getsize(local_path)/1e6:.1f} MB) → Drive")

best = f"{WORK_DIR}/models/model1_complement.pt"
if os.path.exists(best):
    upload_to_drive(best, DRIVE_FOLDER_ID)

print("Done")

In [ ]:
# 9. Confirm Drive folder contents
files = drive_client.ListFile(
    {'q': f"'{DRIVE_FOLDER_ID}' in parents and trashed=false"}
).GetList()

print("Drive folder contents:")
for f in sorted(files, key=lambda x: x['title']):
    size_mb = int(f.get('fileSize', 0)) / 1e6
    print(f"  {f['title']:45s}  {size_mb:7.1f} MB")

---
## Done

`model1_complement.pt` is saved to your Google Drive `musique_data/` folder.

**Next step:** Run `train_kaggle.ipynb` to train Model 2 using the new Model 1 checkpoint.